# 22.1 - Dalla stella al buco nero: versione Google Colab

**Compatibile con Google Colab:** installa automaticamente tutte le librerie richieste. Esegui le celle in ordine; la codifica dei 1.400 frame puo' richiedere alcuni minuti.

**VERSIONE AD ALTA FLUIDITA': 96 frame per la nascita, 1.400 frame per l'accrescimento.**

Questo notebook riprende la fisica introdotta nel `021`, ma la trasforma in una storia animata:

1. una **stella massiccia** consuma l'idrogeno del nucleo;
2. la pressione termica diminuisce e compare la **degenerazione** di elettroni e neutroni;
3. il nucleo supera il limite TOV e curva la griglia dello **spaziotempo** fino all'orizzonte;
4. nasce un **buco nero di Kerr** con disco di accrescimento in rotazione;
5. il flusso magnetico poloidale viene avvolto dal frame dragging e alimenta due jet Blandford-Znajek;
6. un sistema planetario ispirato al Sistema Solare viene perturbato, smembrato e accresciuto.

> **Onesta' del modello.** Le soglie di massa, il raggio di Schwarzschild e il profilo termico del disco sono fisici. Evoluzione stellare, distanze planetarie, dimensioni e tempi sono invece compressi per rendere visibili in pochi secondi fenomeni che avvengono su scale enormemente diverse. Il Sole reale non diventera' un buco nero e i suoi pianeti non cadrebbero automaticamente se fosse sostituito da un buco nero della stessa massa.


## Preparazione automatica di Google Colab

La cella seguente installa tutte le dipendenze necessarie al calcolo numerico e alla codifica MP4. Deve essere eseguita prima delle altre celle.


In [ ]:
import subprocess
import sys

PACCHETTI = [
    'numpy>=1.26',
    'matplotlib>=3.8',
    'imageio-ffmpeg>=0.5',
    'plotly>=5.20',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACCHETTI])

print('Librerie per calcolo e codifica video installate.')


In [ ]:
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display
import imageio_ffmpeg

G = 6.67430e-11
C = 299_792_458.0
M_SOLE = 1.98847e30
M_TERRA = 5.9722e24
M_CHANDRASEKHAR = 1.4
M_TOV = 2.2

plt.rcParams.update({
    'figure.dpi': 105,
    'animation.embed_limit': 250,
    'animation.ffmpeg_path': imageio_ffmpeg.get_ffmpeg_exe(),
    'font.family': 'DejaVu Sans',
})
rng = np.random.default_rng(22)
print('Motore grafico pronto')


## 1. Gli oggetti della simulazione

`Stella` conserva massa, idrogeno, massa del nucleo e stato evolutivo. `Universo` calcola una superficie didattica della curvatura: non e' la metrica completa, ma una visualizzazione del pozzo gravitazionale. `Pianeta` contiene i dati essenziali dei corpi che vedremo orbitare.


In [ ]:
@dataclass
class Stella:
    nome: str
    massa_solari: float
    idrogeno: float = 1.0
    frazione_nucleo: float = 0.13

    @property
    def massa_kg(self):
        return self.massa_solari * M_SOLE

    @property
    def massa_nucleo(self):
        return self.massa_solari * self.frazione_nucleo

    @property
    def idrogeno_kg(self):
        return self.idrogeno * 0.70 * self.massa_kg

    @property
    def raggio_schwarzschild_km(self):
        massa_collassata = self.massa_nucleo * M_SOLE
        return 2 * G * massa_collassata / C**2 / 1000

    def brucia(self, quantita):
        self.idrogeno = max(0.0, self.idrogeno - quantita)
        return self.idrogeno

    def destino(self):
        if self.massa_nucleo < M_CHANDRASEKHAR:
            return 'nana bianca'
        if self.massa_nucleo < M_TOV:
            return 'stella di neutroni'
        return 'buco nero'

    def stato(self, progresso):
        if progresso < 0.42:
            return 'fusione dell idrogeno'
        if progresso < 0.63:
            return 'gigante e combustibili pesanti'
        if progresso < 0.78:
            return 'degenerazione elettronica'
        if progresso < 0.90:
            return 'degenerazione neutronica'
        return 'collasso oltre il limite TOV'


@dataclass(frozen=True)
class Pianeta:
    nome: str
    massa_terre: float
    distanza_au: float
    colore: str
    dimensione: float


class Universo:
    def __init__(self, estensione=10.0, risoluzione=39):
        asse = np.linspace(-estensione, estensione, risoluzione)
        self.x, self.y = np.meshgrid(asse, asse)

    def deformazione(self, compattezza, buco_nero=False):
        r2 = self.x**2 + self.y**2
        profondita = 1.2 + 8.5 * compattezza**2
        morbidezza = max(0.38, 2.8 * (1 - compattezza))
        z = -profondita / np.sqrt(r2 + morbidezza**2)
        if buco_nero:
            z = np.where(r2 < 1.15**2, -11.0, z)
        return z


stella = Stella('Simulazione Black Holes', massa_solari=25.0)
universo = Universo()
pianeti = [
    Pianeta('Mercurio', 0.055, 0.39, '#aaa7a2', 34),
    Pianeta('Venere', 0.815, 0.72, '#e8bd72', 48),
    Pianeta('Terra', 1.000, 1.00, '#55aaff', 50),
    Pianeta('Marte', 0.107, 1.52, '#d96c45', 40),
    Pianeta('Giove', 317.8, 5.20, '#d7aa79', 82),
    Pianeta('Saturno', 95.2, 9.58, '#e9d28f', 73),
    Pianeta('Urano', 14.5, 19.2, '#84d8e8', 60),
    Pianeta('Nettuno', 17.1, 30.1, '#4977db', 58),
    Pianeta('Plutone', 0.0022, 39.5, '#c7b7a3', 32),
    Pianeta('Eris', 0.0028, 67.7, '#d9d7d2', 30),
    Pianeta('Sedna', 0.0002, 506.0, '#b85b42', 28),
]

print(f'Stella: {stella.nome}, {stella.massa_solari:.1f} masse solari')
print(f'Idrogeno iniziale disponibile: {stella.idrogeno_kg:.3e} kg')
print(f'Massa finale del nucleo: {stella.massa_nucleo:.2f} masse solari')
print(f'Destino: {stella.destino().upper()}')
print(f'Raggio di Schwarzschild: {stella.raggio_schwarzschild_km:.1f} km')


## 2. Parametri Kerr, Blandford-Znajek e MAD

Il buco nero non possiede un dipolo permanente: il campo e' sostenuto dal plasma del disco. Il modello seguente usa uno spin di Kerr, un flusso magnetico adimensionale $\phi_{BH}$ e il limite MAD. La potenza e' una stima fisica; forma, densita' e velocita' dei jet restano una visualizzazione cinematica, non una simulazione GRMHD.


In [ ]:
# Parametri modificabili. phi_BH circa 40-60 rappresenta un disco MAD.
A_STAR = 0.90
PHI_BH = 50.0
FRAZIONE_EDDINGTON = 0.10
KAPPA_BZ = 0.05
EFFICIENZA_RADIATIVA = 0.10

if not 0.0 <= A_STAR < 1.0:
    raise ValueError('A_STAR deve essere compreso tra 0 e 1 (1 escluso).')
if PHI_BH < 0 or FRAZIONE_EDDINGTON < 0:
    raise ValueError('Flusso magnetico e tasso di accrescimento devono essere positivi.')

MASSA_BH_KG = stella.massa_nucleo * M_SOLE
R_G_METRI = G * MASSA_BH_KG / C**2
R_H_METRI = R_G_METRI * (1 + np.sqrt(1 - A_STAR**2))
OMEGA_H = A_STAR * C / (2 * R_H_METRI) if A_STAR else 0.0
OMEGA_F = 0.5 * OMEGA_H

# Accrescimento Eddington e correzione di alto spin di Tchekhovskoy et al.
M_PROTONE = 1.67262192369e-27
SIGMA_T = 6.6524587321e-29
L_EDDINGTON = 4*np.pi*G*MASSA_BH_KG*M_PROTONE*C / SIGMA_T
M_DOT_EDDINGTON = L_EDDINGTON / (EFFICIENZA_RADIATIVA*C**2)
M_DOT = FRAZIONE_EDDINGTON * M_DOT_EDDINGTON
X_H = OMEGA_H * R_G_METRI / C
F_CORREZIONE_BZ = 1 + 1.38*X_H**2 - 9.2*X_H**4
ETA_BZ = KAPPA_BZ/(4*np.pi) * PHI_BH**2 * X_H**2 * F_CORREZIONE_BZ
P_BZ_WATT = ETA_BZ * M_DOT * C**2

# Scala grafica. L'orizzonte Kerr e' piu' piccolo di quello di Schwarzschild.
R_H_VIS = 1.75 * R_H_METRI / (2*R_G_METRI)
R_ANELLO_VIS = 2.25
LUNGHEZZA_JET_VIS = 25.0
DELTA_FASE_VISIVA = 0.035
DT_FISICO_PER_FRAME = DELTA_FASE_VISIVA / OMEGA_F if OMEGA_F else 0.0

def geometria_campo3(frame=0, avvolto=True, n_linee=8, n_punti=90):
    # Ogni ramo parte dentro la sfera grafica e attraversa l'orizzonte.
    fase_tempo = OMEGA_F * DT_FISICO_PER_FRAME * frame
    coordinate = []
    for segno in (-1.0, 1.0):
        z0 = 0.55 * R_H_VIS
        z = np.linspace(z0, LUNGHEZZA_JET_VIS, n_punti)
        s = (z-z0)/(LUNGHEZZA_JET_VIS-z0)
        rho = 0.70*R_H_VIS + 3.3*s**0.72
        for phi0 in np.linspace(0, 2*np.pi, n_linee, endpoint=False):
            torsione = segno*4.2*(A_STAR/0.9)*s if avvolto else 0.0
            phi = phi0 + torsione - fase_tempo
            coordinate.append(np.column_stack((rho*np.cos(phi), rho*np.sin(phi), segno*z)))
            coordinate.append(np.full((1, 3), np.nan))
    return np.vstack(coordinate)

print(f'Orizzonte Kerr: {R_H_METRI/1000:.2f} km | a*={A_STAR:.2f}')
print(f'Omega_H={OMEGA_H:.3e} rad/s | Omega_F/Omega_H={OMEGA_F/OMEGA_H if OMEGA_H else 0:.2f}')
print(f'MAD phi_BH={PHI_BH:.0f} | eta_BZ={100*ETA_BZ:.1f}% | P_BZ={P_BZ_WATT:.3e} W')


## 3. Animazione I - nascita del buco nero

La superficie e' la metafora della coperta elastica. Il colore della stella segue la temperatura apparente; i gusci tratteggiati rappresentano, in sequenza, pressione termica, degenerazione elettronica e degenerazione neutronica. Quando nessun guscio riesce piu' a sostenere il nucleo, compare l'orizzonte degli eventi.


In [ ]:
N_NASCITA = 96

fig = plt.figure(figsize=(12, 6.4), facecolor='#030711')
ax = fig.add_subplot(111, projection='3d', facecolor='#030711')
fig.subplots_adjust(left=0.02, right=0.98, bottom=0.08, top=0.88)
idrogeno_txt = fig.text(0.04, 0.94, '', color='#70d6ff', fontsize=11,
                         bbox=dict(boxstyle='round,pad=.4', facecolor='#071527', edgecolor='#247ba0'))
pressione_txt = fig.text(0.72, 0.94, '', color='#ffd166', fontsize=11,
                         bbox=dict(boxstyle='round,pad=.4', facecolor='#211708', edgecolor='#c88719'))

def smoothstep(x):
    x = np.clip(x, 0.0, 1.0)
    return x*x*(3 - 2*x)

def disegna_nascita(frame):
    ax.clear()
    p = frame / (N_NASCITA - 1)
    collasso = smoothstep((p - 0.62) / 0.38)
    buco_nero = p > 0.91
    z = universo.deformazione(0.08 + 0.92*collasso, buco_nero)

    ax.plot_wireframe(universo.x, universo.y, z, rstride=2, cstride=2,
                      color='#2878a8', linewidth=0.55, alpha=0.72)
    ax.contour(universo.x, universo.y, z, levels=10, zdir='z', offset=-11.2,
               cmap='magma', alpha=0.42)

    theta = np.linspace(0, 2*np.pi, 100)
    if not buco_nero:
        r_stella = 2.15*(1 + 0.55*smoothstep((p-0.38)/0.23))*(1-0.78*collasso)
        z0 = -0.4 - 4.5*collasso
        colore = plt.cm.inferno(0.72 - 0.42*collasso)
        ax.scatter([0], [0], [z0], s=1500*r_stella, color=colore,
                   edgecolor='#fff4c2', linewidth=1.2, depthshade=False)
        for scala, stile, nome in [(1.35, '-', 'termica'), (1.72, '--', 'elettroni'),
                                     (2.08, ':', 'neutroni')]:
            visibile = (nome == 'termica' and p < 0.72) or (nome == 'elettroni' and 0.60 < p < 0.85) or (nome == 'neutroni' and p > 0.74)
            if visibile:
                ax.plot(scala*r_stella*np.cos(theta), scala*r_stella*np.sin(theta),
                        np.full_like(theta, z0), stile, color='#dff7ff', lw=1.5, alpha=0.8)
    else:
        ax.scatter([0], [0], [-7.8], s=1700, color='#260914',
                   edgecolor='#ffb13b', linewidth=3.5, depthshade=False)

    idrogeno = max(0.0, 1 - p/0.60)
    stato = stella.stato(p)
    pressione = max(0.0, 1 - smoothstep((p-0.52)/0.39))
    ax.set_title(f'{stella.nome}: {stato.upper()}', color='white', fontsize=15, pad=12, weight='bold')
    idrogeno_txt.set_text(f'Idrogeno nel nucleo  {idrogeno*100:5.1f}%')
    pressione_txt.set_text(f'Sostegno residuo  {pressione*100:5.1f}%')
    ax.set(xlim=(-10, 10), ylim=(-10, 10), zlim=(-11.2, 1.8), xlabel='spazio', ylabel='spazio', zlabel='curvatura')
    ax.view_init(elev=31, azim=35 + 0.22*frame)
    ax.tick_params(colors='#73859b', labelsize=7)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.label.set_color('#8ca0b5')
        axis.pane.set_facecolor((0.01, 0.03, 0.07, 1))
    return []

anim_nascita = FuncAnimation(fig, disegna_nascita, frames=N_NASCITA, interval=70, blit=False)
plt.close(fig)
display(HTML(anim_nascita.to_jshtml(default_mode='once')))


## 4. Animazione II - il buco nero ruota e accresce i pianeti

Il disco usa $T(r) \propto r^{-3/4}$: bianco e giallo indicano la regione piu' calda, rosso quella esterna. Le particelle perdono lentamente momento angolare e spiraleggiano verso l'orizzonte. Per rendere visibile la cattura, il modello applica ai pianeti una forte perturbazione esterna programmata: le orbite reali non decadrebbero in questo modo senza dissipazione o incontri gravitazionali.


In [ ]:
N_FRAME = 1400
FINE_CATTURE = 680
N_DISCO = 850
r0_disco = 2.5 + 7.8*rng.power(1.7, N_DISCO)
phi0_disco = rng.uniform(0, 2*np.pi, N_DISCO)
vel_disco = 0.030*(10/r0_disco)**1.5
deriva = rng.uniform(0.0015, 0.0045, N_DISCO)
r_norm = (r0_disco - r0_disco.min()) / (r0_disco.max() - r0_disco.min())
# Il tempo viscoso cresce con il raggio: il bordo interno cade per primo.
frame_caduta_disco = 35 + 1080*r_norm**1.45 + rng.uniform(0, 90, N_DISCO)
N_DETRITI = 48
fasi_detriti = rng.uniform(0.18, 0.94, (8, N_DETRITI))
jitter_detriti = rng.normal(0, 0.055, (8, N_DETRITI))
temp0 = (r0_disco/2.5)**(-0.75)
cmap_disco = LinearSegmentedColormap.from_list(
    'disco', ['#4d0000', '#b5160b', '#ff6500', '#ffd35a', '#fff9dc', '#d7efff'])
stelle_x = rng.uniform(-24, 24, 260)
stelle_y = rng.uniform(-24, 24, 260)
stelle_s = rng.uniform(0.3, 5.0, 260)
raggi_pianeti = np.concatenate([np.linspace(11.0, 21.5, 8), [25.0, 28.5, 32.0]])
fasi_pianeti = rng.uniform(0, 2*np.pi, len(pianeti))
inizio_cattura = np.concatenate([np.arange(54, 54 + 14*8, 14), [np.inf, np.inf, np.inf]])

fig, ax = plt.subplots(figsize=(10.5, 8.2), facecolor='#01030a')
fig.subplots_adjust(left=0.03, right=0.97, bottom=0.08, top=0.90)
ax.set_facecolor('#01030a')
ax.scatter(stelle_x, stelle_y, s=stelle_s, c='white', alpha=0.42, linewidths=0)
for r_orbita in raggi_pianeti:
    ax.add_patch(plt.Circle((0, 0), r_orbita, fill=False, color='#28435c', lw=0.45, alpha=0.34))

orizzonte = plt.Circle((0, 0), R_H_VIS, color='#260914', ec='#ffb13b', lw=3.0, zorder=10)
anello_fotoni = plt.Circle((0, 0), R_ANELLO_VIS, fill=False, color='#ffe0a3', lw=1.0, alpha=0.75, zorder=9)
ax.add_patch(orizzonte)
ax.add_patch(anello_fotoni)
bagliore = ax.scatter([], [], s=70, c=[], cmap=cmap_disco, vmin=0.2, vmax=1.0, alpha=0.16, linewidths=0)
disco = ax.scatter([], [], s=9, c=[], cmap=cmap_disco, vmin=0.2, vmax=1.0, alpha=0.90, linewidths=0)
detriti = ax.scatter([], [], s=7, c=[], alpha=0.88, linewidths=0, zorder=7)
punti_pianeti = ax.scatter([], [], s=[], c=[], edgecolors='white', linewidths=0.45, zorder=12)
testi = [ax.text(0, 0, '', color='white', fontsize=7.5, zorder=13) for _ in pianeti]
scie = [ax.plot([], [], color=p.colore, lw=1.0, alpha=0.55, zorder=8)[0] for p in pianeti]
stato_txt = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top', color='white', fontsize=10,
                    linespacing=1.35,
                    bbox=dict(boxstyle='round,pad=.50', facecolor='#07111f', edgecolor='#ff8c2a', alpha=.88))
titolo_txt = ax.set_title('Simulazione Black Holes | Evoluzione del sistema', color='white',
                          fontsize=14, weight='bold', pad=14)

def stato_pianeta(tempo_orbita, indice, tempo_cattura=None):
    if tempo_cattura is None:
        tempo_cattura = tempo_orbita
    inizio = inizio_cattura[indice]
    s = smoothstep((tempo_cattura - inizio)/20)
    r = raggi_pianeti[indice]*(1 - 0.91*s)
    omega = 0.009*(22/raggi_pianeti[indice])**1.5
    phi = fasi_pianeti[indice] + omega*tempo_orbita + 4.8*s**2
    return r, phi, s

def aggiorna(frame):
    # Tempo delle catture e tempo orbitale restano separati: le orbite non si fermano.
    t = 176 * min(frame / FINE_CATTURE, 1.0)
    t_rotazione = 176 * frame / FINE_CATTURE
    r_orbitale = np.maximum(r0_disco - deriva*t_rotazione**1.30, 1.78)
    phi = phi0_disco + vel_disco*t_rotazione + 0.38*np.sin(0.045*t_rotazione + r0_disco)
    # Accrescimento viscoso: il tempo di caduta aumenta con il raggio iniziale.
    caduta = smoothstep((frame - frame_caduta_disco) / 105)
    r = 1.70 + (r_orbitale - 1.70)*(1 - caduta)
    vive = caduta < 0.995
    x = np.where(vive, r*np.cos(phi), np.nan)
    y = np.where(vive, 0.62*r*np.sin(phi), np.nan)
    temperatura = np.clip((r/2.25)**(-0.75), 0.2, 1.0)
    disco.set_offsets(np.column_stack([x, y]))
    disco.set_array(temperatura)
    disco.set_sizes(np.where(vive, 9*(1 + 1.4*caduta), 0))
    disco.set_alpha(0.90)
    bagliore.set_offsets(np.column_stack([x, y]))
    bagliore.set_array(temperatura)
    bagliore.set_sizes(np.where(vive, 70*(1 + caduta), 0))
    bagliore.set_alpha(0.16)

    posizioni, colori, dimensioni = [], [], []
    catturati = 0
    for i, pianeta in enumerate(pianeti):
        rp, phip, s = stato_pianeta(t_rotazione, i, t)
        px, py = rp*np.cos(phip), rp*np.sin(phip)
        visibile = s < 0.985
        posizioni.append((px, py) if visibile else (np.nan, np.nan))
        colori.append(pianeta.colore)
        dimensioni.append(pianeta.dimensione*(1 - 0.75*s) if visibile else 0)
        testi[i].set_position((px + 0.35, py + 0.35))
        testi[i].set_text(pianeta.nome if visibile and s < 0.20 else ('spaghettificazione' if visibile else ''))
        frame_scia = np.linspace(max(0, frame-70), frame, 55)
        tempi_orbita = 176*frame_scia/FINE_CATTURE
        tempi_cattura = np.minimum(tempi_orbita, 176)
        trail = np.array([stato_pianeta(to, i, tc)[:2]
                          for to, tc in zip(tempi_orbita, tempi_cattura)])
        tx = trail[:, 0]*np.cos(trail[:, 1])
        ty = trail[:, 0]*np.sin(trail[:, 1])
        scie[i].set_data(tx, ty)
        scie[i].set_alpha(0.55*(1-s))
        catturati += int(s >= 0.985)

    # Ogni pianeta interno viene convertito in una corrente mareale di frammenti.
    dx, dy, dc, ds = [], [], [], []
    for i, pianeta in enumerate(pianeti[:8]):
        q = fasi_detriti[i]
        rilascio = inizio_cattura[i] + 20*q
        eta = smoothstep((t_rotazione - rilascio) / (17 + 10*q))
        attivi = (t_rotazione >= rilascio) & (eta < 0.995)
        r_partenza = raggi_pianeti[i]*(1 - 0.91*q)
        omega = 0.009*(22/raggi_pianeti[i])**1.5
        phi_partenza = fasi_pianeti[i] + omega*rilascio + 4.8*q**2
        r_detrito = 1.70 + (r_partenza - 1.70)*(1 - eta)
        phi_detrito = phi_partenza + (0.08 + 0.42/np.sqrt(r_partenza))*np.maximum(t_rotazione-rilascio, 0)
        phi_detrito += jitter_detriti[i]*(1 + 5*eta)
        dx.extend((r_detrito*np.cos(phi_detrito))[attivi])
        dy.extend((0.62*r_detrito*np.sin(phi_detrito))[attivi])
        dc.extend([pianeta.colore]*int(attivi.sum()))
        ds.extend((5 + 10*eta[attivi]).tolist())
    detriti.set_offsets(np.column_stack([dx, dy]) if dx else np.empty((0, 2)))
    detriti.set_color(dc)
    detriti.set_sizes(ds)

    punti_pianeti.set_offsets(np.array(posizioni))
    punti_pianeti.set_color(colori)
    punti_pianeti.set_sizes(dimensioni)
    massa_acquisita = sum(p.massa_terre for p, start in zip(pianeti, inizio_cattura) if t >= start + 20)
    residue = int(vive.sum())
    in_orbita = len(pianeti) - catturati
    if frame < 300:
        fase = 'Fase 1/3 - il disco interno accresce per primo'
    elif frame < 720:
        fase = 'Fase 2/3 - distruzione mareale dei pianeti'
    else:
        fase = 'Fase 3/3 - orbite esterne e disco residuo'
    titolo_txt.set_text(f'Simulazione Black Holes | {fase}')
    stato_txt.set_text(
        f'Frame {frame+1:04d}/{N_FRAME}  |  orizzonte Kerr {R_H_METRI/1000:.1f} km (non in scala)\n'
        f'Assorbiti: {catturati}  |  in orbita: {in_orbita}  |  disco originario: {residue}/{N_DISCO}\n'
        f'Detriti mareali visibili: {len(dx)}  |  massa accresciuta: {massa_acquisita:,.1f} Terre')
    return [disco, bagliore, detriti, orizzonte, anello_fotoni, punti_pianeti, titolo_txt, stato_txt, *testi, *scie]

ax.set(xlim=(-35, 35), ylim=(-35, 35))
ax.set_aspect('equal')
ax.axis('off')
anim_accrescimento = FuncAnimation(fig, aggiorna, frames=N_FRAME, interval=20, blit=False)
plt.close(fig)
print(f'Codifica MP4 di {N_FRAME:,} frame con controlli pausa e cursore temporale...')
video_html = anim_accrescimento.to_html5_video()
video_html = video_html.replace(
    '<video ', '<video style="display:block;background:#01030a;max-width:100%;height:auto;margin:0" ')
display(HTML(f'<div style="display:inline-block;background:#01030a;line-height:0;padding:0">{video_html}</div>'))


## 5. Esploratore 3D Kerr/BZ per Google Colab

La vista interattiva precalcola **1.400 stati** e li mostra in WebGL. Oltre al disco presenta linee poloidali che attraversano l'orizzonte, il loro avvolgimento toroidale con $\Omega_F=\Omega_H/2$ e plasma nei due jet. Non viene creato un secondo video 3D: Play, Pausa, rotazione, zoom e schermo intero sono disponibili direttamente nell'esploratore.


In [ ]:
# PASSAGGIO 1/2: calcola tutti gli stati prima di creare il grafico.
print(f'Precalcolo di {N_FRAME:,} stati 3D...')

frame_colonna3 = np.arange(N_FRAME, dtype=float)[:, None]
t_catture3 = 176 * np.minimum(frame_colonna3[:, 0] / FINE_CATTURE, 1.0)
t_rotazioni3 = 176 * frame_colonna3[:, 0] / FINE_CATTURE

# Disco: matrici (frame, particella), memorizzate in float32 per contenere la RAM.
r_orb3 = np.maximum(
    r0_disco[None, :] - deriva[None, :] * t_rotazioni3[:, None]**1.30,
    1.78,
)
phi_disco3 = (
    phi0_disco[None, :]
    + vel_disco[None, :] * t_rotazioni3[:, None]
    + 0.38*np.sin(0.045*t_rotazioni3[:, None] + r0_disco[None, :])
)
caduta3 = smoothstep((frame_colonna3 - frame_caduta_disco[None, :]) / 105)
r_disco3 = 1.70 + (r_orb3 - 1.70)*(1 - caduta3)
vive_disco3 = caduta3 < .995
x_disco3 = (r_disco3*np.cos(phi_disco3)).astype(np.float32)
y_disco3 = (r_disco3*np.sin(phi_disco3)).astype(np.float32)
z_disco3 = (0.16*np.sin(3*phi_disco3 + r0_disco[None, :])).astype(np.float32)
temp_disco3 = np.clip((r_disco3/2.25)**(-.75), .2, 1.0).astype(np.float32)
dim_disco3 = (8*(1 + 1.4*caduta3)).astype(np.float32)
residui_disco3 = vive_disco3.sum(axis=1)

# Pianeti: posizioni, dimensioni e conteggi per ogni frame.
n_pianeti3 = len(pianeti)
pos_pianeti3 = np.full((N_FRAME, n_pianeti3, 3), np.nan, dtype=np.float32)
dim_pianeti3 = np.zeros((N_FRAME, n_pianeti3), dtype=np.float32)
catturati3 = np.zeros(N_FRAME, dtype=np.int16)
for frame in range(N_FRAME):
    t = t_catture3[frame]
    t_rot = t_rotazioni3[frame]
    for i, pianeta in enumerate(pianeti):
        rp, pp, s = stato_pianeta(t_rot, i, t)
        if s < .985:
            pos_pianeti3[frame, i] = (
                rp*np.cos(pp), rp*np.sin(pp), .25*np.sin(2*pp)
            )
            dim_pianeti3[frame, i] = pianeta.dimensione*(1-.75*s)
        catturati3[frame] += int(s >= .985)

# Detriti: il numero cambia nel tempo, quindi ogni frame conserva un array compatto.
pos_detriti3 = []
col_detriti3 = []
dim_detriti3 = []
for frame in range(N_FRAME):
    t_rot = t_rotazioni3[frame]
    fx, fy, fz, fc, fs = [], [], [], [], []
    for i, pianeta in enumerate(pianeti[:8]):
        q = fasi_detriti[i]
        rilascio = inizio_cattura[i] + 20*q
        eta = smoothstep((t_rot-rilascio)/(17+10*q))
        attivi = (t_rot >= rilascio) & (eta < .995)
        r0f = raggi_pianeti[i]*(1-.91*q)
        omega = .009*(22/raggi_pianeti[i])**1.5
        p0f = fasi_pianeti[i] + omega*rilascio + 4.8*q**2
        rf = 1.70 + (r0f-1.70)*(1-eta)
        pf = p0f + (.08+.42/np.sqrt(r0f))*np.maximum(t_rot-rilascio, 0)
        pf += jitter_detriti[i]*(1+5*eta)
        fx.extend((rf*np.cos(pf))[attivi])
        fy.extend((rf*np.sin(pf))[attivi])
        fz.extend((.10*np.sin(5*pf))[attivi])
        fc.extend([pianeta.colore]*int(attivi.sum()))
        fs.extend((5+10*eta[attivi]).tolist())
    if fx:
        pos_detriti3.append(np.column_stack([fx, fy, fz]).astype(np.float32))
        dim_detriti3.append(np.asarray(fs, dtype=np.float32))
    else:
        pos_detriti3.append(np.empty((0, 3), dtype=np.float32))
        dim_detriti3.append(np.empty(0, dtype=np.float32))
    col_detriti3.append(fc)

# Plasma dei due jet: moto relativistico compresso nel tempo e guidato da Omega_F.
N_JET3 = 180
rng_jet3 = np.random.default_rng(2202)
segno_jet3 = np.where(np.arange(N_JET3) % 2 == 0, 1.0, -1.0)
fase_lancio_jet3 = rng_jet3.random(N_JET3)
phi0_jet3 = rng_jet3.uniform(0, 2*np.pi, N_JET3)
cicli_jet3 = 5.0*np.arange(N_FRAME, dtype=float)[:, None]/max(N_FRAME-1, 1)
eta_jet3 = (cicli_jet3 - fase_lancio_jet3[None, :]) % 1.0
z_jet3 = segno_jet3[None, :] * (R_H_VIS + eta_jet3*(LUNGHEZZA_JET_VIS-R_H_VIS))
rho_jet3 = 0.16 + 0.055*np.abs(z_jet3) + 0.10*np.sin(7*eta_jet3)**2
fase_campo3 = OMEGA_F*DT_FISICO_PER_FRAME*np.arange(N_FRAME, dtype=float)[:, None]
phi_jet3 = (phi0_jet3[None, :]
            + segno_jet3[None, :]*5*(A_STAR/0.9)*eta_jet3 - fase_campo3)
pos_jet3 = np.stack((
    rho_jet3*np.cos(phi_jet3), rho_jet3*np.sin(phi_jet3), z_jet3
), axis=2).astype(np.float32)
if P_BZ_WATT == 0:
    pos_jet3[:] = np.nan
lum_jet3 = (1.0 - 0.62*eta_jet3).astype(np.float32)

# Le grandi matrici temporanee non servono durante il rendering.
del frame_colonna3, r_orb3, phi_disco3, caduta3, r_disco3
del cicli_jet3, eta_jet3, rho_jet3, fase_campo3, phi_jet3, z_jet3
memoria_mb3 = sum(a.nbytes for a in (
    x_disco3, y_disco3, z_disco3, temp_disco3, dim_disco3,
    vive_disco3, pos_pianeti3, dim_pianeti3, pos_jet3, lum_jet3,
)) / 1024**2
print(f'Precalcolo completato. Dati principali in RAM: {memoria_mb3:.1f} MB')
print('Ora puoi aprire l esploratore 3D interattivo.')


### Esploratore 3D interattivo (opzionale)

Questa vista usa gli stati gia' precalcolati. Puoi avviare o fermare il tempo, trascinare per cambiare punto di vista, usare la rotellina per lo zoom e il pulsante **Schermo intero**. I pianeti integri mantengono il proprio nome; il disco usa un gradiente dal rosso esterno al bianco-azzurro interno. Al termine viene creato anche un file HTML autonomo e scaricabile.


In [ ]:
# Esploratore WebGL: Play/Pausa, rotazione e zoom senza ricalcolare la fisica.
import plotly.graph_objects as go

PASSO_INTERATTIVO_3D = 5
frame_interattivi3 = list(range(0, N_FRAME, PASSO_INTERATTIVO_3D))
if frame_interattivi3[-1] != N_FRAME - 1:
    frame_interattivi3.append(N_FRAME - 1)

def tracce_interattive3(frame):
    vive = vive_disco3[frame]
    visibili = dim_pianeti3[frame] > 0
    pp = pos_pianeti3[frame, visibili]
    fd = pos_detriti3[frame]
    campo = geometria_campo3(frame, avvolto=True)
    jet = pos_jet3[frame]
    nord = segno_jet3 > 0
    raggio_visibile = np.hypot(x_disco3[frame, vive], y_disco3[frame, vive])
    colore_radiale = np.clip((10.5-raggio_visibile)/(10.5-R_H_VIS), 0, 1)
    return [
        go.Scatter3d(
            x=x_disco3[frame, vive], y=y_disco3[frame, vive],
            z=z_disco3[frame, vive], mode='markers', name='Disco',
            marker=dict(
                size=np.maximum(dim_disco3[frame, vive]/2.2, 3.4),
                color=colore_radiale,
                colorscale=[[0, '#7a0606'], [.28, '#d42b0b'], [.55, '#ff7a16'],
                            [.78, '#ffd866'], [.93, '#fff9dc'], [1, '#d8f4ff']],
                cmin=0, cmax=1, opacity=.95,
                line=dict(color='rgba(255,210,125,.30)', width=.35),
            ),
            hoverinfo='skip',
        ),
        go.Scatter3d(
            x=fd[:, 0], y=fd[:, 1], z=fd[:, 2],
            mode='markers', name='Detriti',
            marker=dict(
                size=np.maximum(dim_detriti3[frame]/2.0, 3.5),
                color=col_detriti3[frame], opacity=.96,
                line=dict(color='rgba(255,255,255,.35)', width=.4),
            ),
            hoverinfo='skip',
        ),
        go.Scatter3d(
            x=pp[:, 0], y=pp[:, 1], z=pp[:, 2],
            mode='markers+text', name='Pianeti',
            marker=dict(
                size=np.maximum(dim_pianeti3[frame, visibili]/6, 6),
                color=[p.colore for p, ok in zip(pianeti, visibili) if ok],
                line=dict(color='white', width=1), opacity=.96,
            ),
            text=[p.nome for p, ok in zip(pianeti, visibili) if ok],
            textposition='top center',
            textfont=dict(color='#ffffff', size=11, family='Arial Black'),
            hovertemplate='%{text}<extra></extra>',
        ),
        go.Scatter3d(
            x=campo[:, 0], y=campo[:, 1], z=campo[:, 2],
            mode='lines', name='Campo toroidale',
            line=dict(color='#58d9ff', width=3), opacity=.72, hoverinfo='skip',
        ),
        go.Scatter3d(
            x=jet[nord, 0], y=jet[nord, 1], z=jet[nord, 2],
            mode='markers', name='Jet nord',
            marker=dict(size=3.2, color=lum_jet3[frame, nord],
                        colorscale=[[0, '#1261a0'], [1, '#d9fbff']],
                        cmin=0, cmax=1, opacity=.78), hoverinfo='skip',
        ),
        go.Scatter3d(
            x=jet[~nord, 0], y=jet[~nord, 1], z=jet[~nord, 2],
            mode='markers', name='Jet sud',
            marker=dict(size=3.2, color=lum_jet3[frame, ~nord],
                        colorscale=[[0, '#1261a0'], [1, '#d9fbff']],
                        cmin=0, cmax=1, opacity=.78), hoverinfo='skip',
        ),
    ]

# Sfondo, linee poloidali, orizzonte e anello sono statici.
rng_stelle_plot3 = np.random.default_rng(2201)
n_stelle_plot3 = 360
direzioni_stelle3 = rng_stelle_plot3.normal(size=(n_stelle_plot3, 3))
direzioni_stelle3 /= np.linalg.norm(direzioni_stelle3, axis=1)[:, None]
raggi_stelle3 = rng_stelle_plot3.uniform(27, 34, n_stelle_plot3)
stelle_plot3 = direzioni_stelle3 * raggi_stelle3[:, None]
u_plot3 = np.linspace(0, 2*np.pi, 28)
v_plot3 = np.linspace(0, np.pi, 14)
xs_plot3 = R_H_VIS*np.outer(np.cos(u_plot3), np.sin(v_plot3))
ys_plot3 = R_H_VIS*np.outer(np.sin(u_plot3), np.sin(v_plot3))
zs_plot3 = R_H_VIS*np.outer(np.ones_like(u_plot3), np.cos(v_plot3))
angolo_plot3 = np.linspace(0, 2*np.pi, 150)
campo_poloidale_plot3 = geometria_campo3(0, avvolto=False)
statiche3 = [
    go.Scatter3d(
        x=stelle_plot3[:, 0], y=stelle_plot3[:, 1], z=stelle_plot3[:, 2],
        mode='markers', name='Campo stellare',
        marker=dict(size=rng_stelle_plot3.uniform(1, 3.5, n_stelle_plot3),
                    color='white', opacity=.62),
        hoverinfo='skip', showlegend=False,
    ),
    go.Scatter3d(
        x=campo_poloidale_plot3[:, 0], y=campo_poloidale_plot3[:, 1],
        z=campo_poloidale_plot3[:, 2], mode='lines', name='Campo poloidale',
        line=dict(color='#3a7da0', width=2), opacity=.34, hoverinfo='skip',
    ),
    go.Surface(
        x=xs_plot3, y=ys_plot3, z=zs_plot3,
        colorscale=[[0, '#17030d'], [.50, '#350817'], [1, '#7a2518']], showscale=False,
        opacity=.98, name='Orizzonte Kerr', hoverinfo='skip',
        lighting=dict(ambient=.72, diffuse=.85, specular=.45, roughness=.72),
        lightposition=dict(x=20, y=-25, z=35),
    ),
    go.Scatter3d(
        x=R_ANELLO_VIS*np.cos(angolo_plot3), y=R_ANELLO_VIS*np.sin(angolo_plot3),
        z=np.zeros_like(angolo_plot3), mode='lines', name='Anello fotonico',
        line=dict(color='#ffd27a', width=7), hoverinfo='skip',
    ),
]

tracce_iniziali3 = tracce_interattive3(frame_interattivi3[0])
fotogrammi_plot3 = [
    go.Frame(
        name=str(frame), data=tracce_interattive3(frame), traces=list(range(6))
    )
    for frame in frame_interattivi3
]

passi_slider3 = [
    dict(
        method='animate',
        label=str(frame) if frame % 100 == 0 or frame == N_FRAME-1 else '',
        args=[[str(frame)], dict(
            mode='immediate',
            frame=dict(duration=0, redraw=True),
            transition=dict(duration=0),
        )],
    )
    for frame in frame_interattivi3
]

fig_interattiva3 = go.Figure(
    data=tracce_iniziali3 + statiche3,
    frames=fotogrammi_plot3,
)
fig_interattiva3.update_layout(
    title=dict(
        text='<b>BUCO NERO DI KERR + JET BZ</b>',
        x=.5, y=.985, xanchor='center', yanchor='top',
        font=dict(size=22, color='#fff4dc'), pad=dict(t=2, b=4),
    ),
    paper_bgcolor='#01030a', plot_bgcolor='#01030a',
    font=dict(color='#f3eadc', family='Arial'), height=820,
    margin=dict(l=12, r=12, t=82, b=25),
    modebar=dict(
        orientation='v', bgcolor='rgba(24,12,5,.90)',
        color='#d9c4a5', activecolor='#ff9d2e',
    ),
    uirevision='mantieni-camera',
    scene=dict(
        bgcolor='#01030a',
        xaxis=dict(visible=False, range=[-35, 35]),
        yaxis=dict(visible=False, range=[-35, 35]),
        zaxis=dict(visible=False, range=[-27, 27]),
        aspectmode='manual', aspectratio=dict(x=1, y=1, z=.78),
        camera=dict(eye=dict(x=1.35, y=-1.55, z=.75)),
    ),
    updatemenus=[dict(
        type='buttons', direction='left',
        x=.015, xanchor='left', y=.985, yanchor='top',
        bgcolor='rgba(24,12,5,.88)', bordercolor='#ff9d2e', borderwidth=1,
        font=dict(size=14, color='#fff4dc'), pad=dict(l=7, r=7, t=7, b=7),
        showactive=False,
        buttons=[
            dict(
                label='&#9654;&nbsp; Play', method='animate',
                args=[None, dict(
                    fromcurrent=True, mode='immediate',
                    frame=dict(duration=120, redraw=True),
                    transition=dict(duration=0),
                )],
            ),
            dict(
                label='&#10074;&#10074;&nbsp; Pausa', method='animate',
                args=[[None], dict(
                    mode='immediate',
                    frame=dict(duration=0, redraw=False),
                    transition=dict(duration=0),
                )],
            ),
        ],
    )],
    sliders=[dict(
        active=0, steps=passi_slider3, x=.08, len=.84,
        bgcolor='rgba(20,10,4,.72)', bordercolor='#7f4a1f',
        currentvalue=dict(visible=False),
        transition=dict(duration=0),
    )],
)

CONFIG_ESPLORATORE_3D = {
    'scrollZoom': True,
    'responsive': True,
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToRemove': ['hoverClosest3d', 'toggleSpikelines'],
    'toImageButtonOptions': {
        'format': 'png', 'filename': 'simulazione_black_holes',
        'height': 900, 'width': 1400, 'scale': 1
    },
}
SCRIPT_SCHERMO_INTERO3 = r'''
const grafico = document.getElementById('{plot_id}');
grafico.style.position = 'relative';
const pulsante = document.createElement('button');
pulsante.textContent = 'Schermo intero';
pulsante.title = 'Apri o chiudi lo schermo intero';
pulsante.style.cssText = 'position:absolute;right:18px;bottom:18px;z-index:20;padding:11px 16px;' +
  'border:1px solid #ff9d2e;border-radius:5px;background:rgba(24,12,5,.92);' +
  'color:#fff4dc;font:600 13px Arial;cursor:pointer';
pulsante.onclick = () => {
  if (!document.fullscreenElement) {
    (grafico.requestFullscreen || grafico.webkitRequestFullscreen).call(grafico);
  } else {
    (document.exitFullscreen || document.webkitExitFullscreen).call(document);
  }
};
document.addEventListener('fullscreenchange', () => {
  const attivo = document.fullscreenElement === grafico;
  grafico.style.width = attivo ? '100vw' : '100%';
  grafico.style.height = attivo ? '100vh' : '820px';
  grafico.style.background = '#01030a';
  pulsante.textContent = attivo ? 'Esci da schermo intero' : 'Schermo intero';
  Plotly.Plots.resize(grafico);
});
grafico.appendChild(pulsante);
'''
PERCORSO_ESPLORATORE_3D = '/content/simulazione_black_holes_interattiva.html'
fig_interattiva3.write_html(
    PERCORSO_ESPLORATORE_3D, include_plotlyjs=True, full_html=True,
    auto_play=False, config=CONFIG_ESPLORATORE_3D,
    post_script=SCRIPT_SCHERMO_INTERO3,
)
print(f'Esploratore pronto: {len(frame_interattivi3)} stati WebGL.')
print(f'File interattivo creato: {PERCORSO_ESPLORATORE_3D}')
html_interattivo3 = fig_interattiva3.to_html(
    include_plotlyjs='cdn', full_html=False, auto_play=False,
    config=CONFIG_ESPLORATORE_3D, post_script=SCRIPT_SCHERMO_INTERO3,
)
from html import escape as escape_html
cornice_interattiva3 = (
    '<iframe allowfullscreen allow="fullscreen" '    'style="display:block;width:100%;height:860px;border:0;background:#01030a" '    'srcdoc="' + escape_html(html_interattivo3, quote=True) + '"></iframe>'
)
display(HTML(cornice_interattiva3))


### Download dell'esploratore interattivo

Esegui la cella seguente dopo aver creato l'esploratore. Il file HTML funziona anche fuori da Colab e conserva Play, Pausa, rotazione, zoom e schermo intero.


In [ ]:
from google.colab import files
files.download(PERCORSO_ESPLORATORE_3D)


## 6. Come leggere quello che hai visto

- **Idrogeno:** e' il combustibile iniziale; la stella non collassa perche' l'idrogeno pesa meno, ma perche' la fusione non produce piu' pressione sufficiente.
- **Degenerazione:** non significa che gli atomi diventano degenerati in senso comune. E' una pressione quantistica dovuta al principio di esclusione di Pauli. Prima resistono gli elettroni, poi i neutroni.
- **Limite TOV:** il nucleo della simulazione vale 3.25 masse solari, oltre i circa 2.2 usati dal modello; per questo nessuna pressione nota arresta il collasso.
- **Griglia:** e' una metafora bidimensionale immersa in 3D. Lo spaziotempo reale ha quattro dimensioni e non sprofonda materialmente verso il basso.
- **Disco:** il colore segue il profilo di Shakura-Sunyaev $T \propto r^{-3/4}$ e la rotazione e' piu' rapida vicino al centro. Il tempo viscoso aumenta con il raggio, quindi le particelle interne vengono accresciute prima di quelle esterne.
- **Peli magnetici:** le linee azzurre non sono materia e non violano il teorema no-hair. Sono campo poloidale sostenuto dal plasma, disegnato fino a punti interni alla superficie grafica per rendere visibile l'attraversamento dell'orizzonte.
- **Blandford-Znajek:** il frame dragging avvolge il campo. Le linee brillanti ruotano con $\Omega_F=\Omega_H/2$ e il plasma viene guidato nei due jet; il moto e' compresso nel tempo ma conserva questo rapporto.
- **MAD e potenza:** $\phi_{BH}=50$ rappresenta una configurazione magnetically arrested. $P_{BZ}$ e' stimata da spin, flusso e tasso di accrescimento; non e' ricavata dinamicamente dalle particelle grafiche del disco.
- **Distruzione mareale:** un pianeta catturato non sparisce come un punto. Superato il limite mareale viene allungato, frammentato e trasformato in una corrente di detriti; i frammenti acquistano velocita' orbitale mentre spiraleggiano verso l'orizzonte.
- **Corpi lontani:** Plutone, Eris e Sedna non ricevono dissipazione nel modello e continuano a orbitare. La formazione del buco nero non implica automaticamente la caduta di tutti i corpi del sistema.
- **Limite del modello:** tempi di cattura, dissipazione, collimazione e densita' dei jet sono prescritti. La potenza BZ usa una parametrizzazione fisica, ma una previsione quantitativa della struttura richiederebbe un codice GRMHD.

### Esperimenti

Cambia `A_STAR`, `PHI_BH` e `FRAZIONE_EDDINGTON`, poi riesegui dal blocco Kerr/BZ in avanti. Con `A_STAR=0` la potenza e l'avvolgimento si annullano; raddoppiando `PHI_BH`, la potenza cresce di circa quattro volte. Aumenta `N_DISCO` per un anello piu' denso.
